# 40 - Gold: County Price Trends

**Purpose:** Analytics-ready aggregations for Syracuse MSA housing market

**Layer:** Gold (business-ready, aggregated insights)

**Source:** Silver `home_prices` table (FHFA HPI + Redfin + Realtor.com)

**Output:** `workspace.gold.county_price_trends` - Monthly housing metrics by county

**Grain:** One row per county per month

**Key Metrics:**
* Median sale price (from Redfin)
* Median list price (from Redfin & Realtor.com)
* House Price Index (from FHFA)
* Year-over-year price changes
* Month-over-month trends

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

CATALOG = "workspace"
GOLD_SCHEMA = "gold"
SILVER_SCHEMA = "silver"

# County name mapping for Syracuse MSA
COUNTY_NAMES = {
    "36067": "Onondaga County",
    "36053": "Madison County",
    "36075": "Oswego County"
}

In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}")
print(f"✓ Schema {CATALOG}.{GOLD_SCHEMA} ready")

In [0]:
# Read silver home_prices and aggregate by county + month
# Each source provides different metrics, so we'll coalesce them

home_prices = spark.table(f"{CATALOG}.{SILVER_SCHEMA}.home_prices")

# Aggregate by county and month, taking the best value from each source
county_month = (
    home_prices
    .groupBy("county_fips", "date_key")
    .agg(
        # Redfin provides sale prices
        F.max(F.when(F.col("source") == "redfin", F.col("median_sale_price"))).alias("median_sale_price"),
        
        # Both Redfin and Realtor.com provide list prices - take average
        F.avg(F.col("median_list_price")).alias("median_list_price"),
        
        # FHFA provides HPI index
        F.max(F.when(F.col("source") == "fhfa", F.col("hpi_index"))).alias("hpi_index"),
        F.max(F.when(F.col("source") == "fhfa", F.col("hpi_annual_change_pct"))).alias("hpi_annual_change_pct")
    )
    .withColumn("period_month", F.col("date_key"))  # Rename for clarity
    .drop("date_key")
)

print(f"✓ Aggregated to {county_month.count()} county-month records")

In [0]:
# Add county names
county_name_expr = F.create_map(
    *[x for pair in [(F.lit(k), F.lit(v)) for k, v in COUNTY_NAMES.items()] for x in pair]
)

county_trends = (
    county_month
    .withColumn("county_name", county_name_expr[F.col("county_fips")])
    .withColumn("msa", F.lit("Syracuse, NY MSA"))
)

# Calculate month-over-month changes
window_spec = Window.partitionBy("county_fips").orderBy("period_month")

county_trends = (
    county_trends
    .withColumn("prev_sale_price", F.lag("median_sale_price", 1).over(window_spec))
    .withColumn("prev_list_price", F.lag("median_list_price", 1).over(window_spec))
    .withColumn(
        "sale_price_mom_pct",
        ((F.col("median_sale_price") - F.col("prev_sale_price")) / F.col("prev_sale_price") * 100)
    )
    .withColumn(
        "list_price_mom_pct",
        ((F.col("median_list_price") - F.col("prev_list_price")) / F.col("prev_list_price") * 100)
    )
    .drop("prev_sale_price", "prev_list_price")
)

# Calculate year-over-year changes
window_spec_yoy = Window.partitionBy("county_fips").orderBy("period_month")

county_trends = (
    county_trends
    .withColumn("prev_year_sale_price", F.lag("median_sale_price", 12).over(window_spec_yoy))
    .withColumn(
        "sale_price_yoy_pct",
        ((F.col("median_sale_price") - F.col("prev_year_sale_price")) / F.col("prev_year_sale_price") * 100)
    )
    .drop("prev_year_sale_price")
)

print(f"✓ Added calculated metrics (MoM and YoY changes)")

In [0]:
# Select final columns and add metadata
final_columns = [
    "county_fips",
    "county_name",
    "msa",
    "period_month",
    "median_sale_price",
    "median_list_price",
    "hpi_index",
    "hpi_annual_change_pct",
    "sale_price_mom_pct",
    "list_price_mom_pct",
    "sale_price_yoy_pct"
]

gold_table = (
    county_trends
    .select(*final_columns)
    .withColumn("_updated_at", F.current_timestamp())
)

# Write to gold layer
gold_table.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable(f"{CATALOG}.{GOLD_SCHEMA}.county_price_trends")

print(f"\n✅ Gold table created: {CATALOG}.{GOLD_SCHEMA}.county_price_trends")
print(f"   Total rows: {gold_table.count()}")

In [0]:
# Show the most recent month for all counties
print("\n📊 Latest Month - Syracuse MSA Housing Trends:\n")

latest = spark.table(f"{CATALOG}.{GOLD_SCHEMA}.county_price_trends") \
    .orderBy(F.col("period_month").desc()) \
    .limit(3)

display(latest)

## Gold Layer Summary

The `county_price_trends` table provides:

* **Grain**: One row per county per month
* **Time Range**: 2020-01 to 2024-12 (60 months)
* **Counties**: 3 (Onondaga, Madison, Oswego)
* **Total Rows**: 180

**Key Metrics Available:**
* Median sale and list prices
* House Price Index (HPI)
* Month-over-month % changes
* Year-over-year % changes
* Annual HPI growth rate

**Next Steps:**
* Use this table for dashboards and analytics
* The metric view (30_metric_view) can now reference this table
* Create visualizations showing price trends over time